# 🛣️ RoadGuard: Deep Learning Road Damage Detection (RDD2022 Training Pipeline)
### Complete Google Colab GPU Training Workflow

This notebook trains and evaluates real deep learning models for **RoadGuard** on the official **RDD2022** dataset.

#### Classes Detected:
- `D00`: Longitudinal Crack
- `D10`: Transverse Crack
- `D20`: Alligator Crack (Structural Fatigue)
- `D40`: Pothole

---

## 1. Environment & GPU Verification

In [ ]:
# Verify GPU availability (Recommended: Tesla T4 or A100)
!nvidia-smi

# Install modern Ultralytics & dependencies
!pip install --upgrade ultralytics opencv-python-headless pyyaml matplotlib

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device  : {torch.cuda.get_device_name(0)}")

## 2. Download RDD2022 Dataset Subsets
We download the official CRDDC'2022 dataset archives directly into the Colab environment.

In [ ]:
import os
import urllib.request
import tarfile
from pathlib import Path

DATA_DIR = Path("/content/rdd2022_raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Official RDD2022 Japan & India Subsets
URLS = {
    "Japan": "https://bigdata-cup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022_Japan.tar.gz",
    "India": "https://bigdata-cup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022_India.tar.gz"
}

for country, url in URLS.items():
    archive = DATA_DIR / f"{country}.tar.gz"
    if not archive.exists():
        print(f"Downloading {country} subset...")
        !wget -q --show-progress {url} -O {archive}
        print(f"Extracting {country}...")
        !tar -xzf {archive} -C {DATA_DIR}

print("Extraction complete!")
!ls -la /content/rdd2022_raw

## 3. Convert Pascal VOC XML Annotations to YOLO Format

In [ ]:
import xml.etree.ElementTree as ET
import glob
from tqdm import tqdm

CLASS_MAP = {
    'D00': 0, # Longitudinal Crack
    'D10': 1, # Transverse Crack
    'D20': 2, # Alligator Crack
    'D40': 3  # Pothole
}

def convert_voc_bbox(size, box):
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    xmin, ymin, xmax, ymax = box
    x_center = ((xmin + xmax) / 2.0) * dw
    y_center = ((ymin + ymax) / 2.0) * dh
    width = (xmax - xmin) * dw
    height = (ymax - ymin) * dh
    return round(x_center, 6), round(y_center, 6), round(width, 6), round(height, 6)

def process_xml(xml_path, out_txt_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)
    if w <= 0 or h <= 0: return False

    lines = []
    for obj in root.findall('object'):
        cls_name = obj.find('name').text.strip()
        matched = None
        for k, v in CLASS_MAP.items():
            if cls_name == k or cls_name.startswith(k):
                matched = v
                break
        if matched is None: continue
        bnd = obj.find('bndbox')
        xmin = float(bnd.find('xmin').text)
        ymin = float(bnd.find('ymin').text)
        xmax = float(bnd.find('xmax').text)
        ymax = float(bnd.find('ymax').text)
        if xmax <= xmin or ymax <= ymin: continue
        box = convert_voc_bbox((w, h), (xmin, ymin, xmax, ymax))
        lines.append(f"{matched} {box[0]} {box[1]} {box[2]} {box[3]}")

    if lines:
        Path(out_txt_path).parent.mkdir(parents=True, exist_ok=True)
        with open(out_txt_path, 'w') as f:
            f.write('\n'.join(lines))
        return True
    return False

print("XML converter defined.")

## 4. Build Dataset Splits & YAML Configuration

In [ ]:
import random
import shutil

DATASET_ROOT = Path('/content/rdd2022_yolo')
for split in ['train', 'val', 'test']:
    (DATASET_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Create YAML configuration
yaml_content = f"""
path: /content/rdd2022_yolo
train: images/train
val: images/val
test: images/test

names:
  0: D00_Longitudinal_Crack
  1: D10_Transverse_Crack
  2: D20_Alligator_Crack
  3: D40_Pothole
"""

with open('/content/rdd2022.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("Dataset config saved to /content/rdd2022.yaml")

## 5. Train Baseline Model (YOLOv8n)
Train lightweight baseline detector suitable for deployment.

In [ ]:
from ultralytics import YOLO

# Initialize pretrained YOLOv8n
baseline_model = YOLO('yolov8n.pt')

# Train for 50-100 epochs
results_baseline = baseline_model.train(
    data='/content/rdd2022.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    patience=15,
    project='runs/train',
    name='rdd2022_yolov8n_baseline'
)

## 6. Train Improved Model (Domain-Adapted for Small Cracks)
Trains with tuned crack-loss weighting and mosaic augmentations.

In [ ]:
improved_model = YOLO('yolov8s.pt')

results_improved = improved_model.train(
    data='/content/rdd2022.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    box=8.0,
    cls=0.7,
    dfl=1.8,
    patience=20,
    project='runs/train',
    name='rdd2022_yolov8s_improved'
)

## 7. Comparative Model Evaluation & Metrics

In [ ]:
# Validate on validation split
val_metrics = baseline_model.val(data='/content/rdd2022.yaml', split='val')

print("=== Baseline Validation Metrics ===")
print(f"mAP@0.5      : {val_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {val_metrics.box.map:.4f}")
print(f"Precision    : {val_metrics.box.mp:.4f}")
print(f"Recall       : {val_metrics.box.mr:.4f}")

## 8. Export Weights for RoadGuard FastAPI Integration
Download the best model checkpoint to transfer into your local `RoadGuard/ml/models/roadguard_yolov8n.pt`.

In [ ]:
from google.colab import files
import shutil

best_checkpoint = '/content/runs/train/rdd2022_yolov8n_baseline/weights/best.pt'
target_name = '/content/roadguard_yolov8n.pt'
shutil.copy2(best_checkpoint, target_name)

print(f"Ready for download: {target_name}")
files.download(target_name)